#IMDB dataset

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [3]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Conv1D

vocab_size = 10000
max_length = 50
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)

x_train = pad_sequences(x_train, maxlen=max_length)
x_test = pad_sequences(x_test, maxlen=max_length)

glove_file = '/content/gdrive/MyDrive/glove.6B/glove.6B.100d.txt'
embedding_dim = 100

embeddings_index = {}
with open(glove_file, 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

embedding_matrix = np.zeros((vocab_size, embedding_dim))
word_index = imdb.get_word_index()
for word, i in word_index.items():
    if i < vocab_size:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector

model = Sequential()
model.add(Embedding(vocab_size, embedding_dim, input_length=max_length, weights=[embedding_matrix], trainable=False))
model.add(Conv1D(16, 5, activation='relu'))
model.add(LSTM(32, return_sequences=True))
model.add(LSTM(64))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.fit(x_train, y_train, epochs=10, batch_size=64, validation_data=(x_test, y_test))

loss, accuracy = model.evaluate(x_test, y_test)
print(f"Final Loss: {loss:.4f}, Final Accuracy: {accuracy:.4f}")

y_pred = (model.predict(x_test) > 0.5).astype("int32")

cm = confusion_matrix(y_test, y_pred)
precision = cm[1, 1] / (cm[1, 1] + cm[0, 1])
recall = cm[1, 1] / (cm[1, 1] + cm[1, 0])

print("Confusion Matrix:")
print(cm)
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

Epoch 1/10
391/391 [==============================] - 43s 100ms/step - loss: 0.6860 - accuracy: 0.5427 - val_loss: 0.6818 - val_accuracy: 0.5838
Epoch 2/10
391/391 [==============================] - 34s 87ms/step - loss: 0.6553 - accuracy: 0.6103 - val_loss: 0.6380 - val_accuracy: 0.6353
Epoch 3/10
391/391 [==============================] - 35s 89ms/step - loss: 0.6173 - accuracy: 0.6514 - val_loss: 0.6275 - val_accuracy: 0.6445
Epoch 4/10
391/391 [==============================] - 36s 91ms/step - loss: 0.5731 - accuracy: 0.6926 - val_loss: 0.6082 - val_accuracy: 0.6690
Epoch 5/10
391/391 [==============================] - 34s 88ms/step - loss: 0.5411 - accuracy: 0.7193 - val_loss: 0.6035 - val_accuracy: 0.6691
Epoch 6/10
391/391 [==============================] - 34s 86ms/step - loss: 0.5181 - accuracy: 0.7374 - val_loss: 0.6052 - val_accuracy: 0.6834
Epoch 7/10
391/391 [==============================] - 35s 91ms/step - loss: 0.4962 - accuracy: 0.7535 - val_loss: 0.5897 - val_accuracy

#Persian Dataset

In [5]:
!pip install datasets
from datasets import load_dataset

In [6]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, LSTM, Dense
from sklearn.metrics import confusion_matrix, precision_score, recall_score
import numpy as np
from datasets import load_dataset

persian_dataset = load_dataset("SeyedAli/Persian-Text-Sentiment")

glove_embeddings_index = {}
with open('/content/gdrive/MyDrive/glove.6B/glove.6B.100d.txt', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        glove_embeddings_index[word] = coefs

tokenizer = Tokenizer()
tokenizer.fit_on_texts(persian_dataset['train']['text'])
sequences = tokenizer.texts_to_sequences(persian_dataset['train']['text'])

maxlen = 100
X = pad_sequences(sequences, maxlen=maxlen)
y = np.array(persian_dataset['train']['label'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

embedding_dim = 100
word_index = tokenizer.word_index
embedding_matrix = np.zeros((len(word_index) + 1, embedding_dim))
for word, i in word_index.items():
    embedding_vector = glove_embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

model = Sequential()
model.add(Embedding(len(word_index) + 1, embedding_dim, weights=[embedding_matrix], input_length=maxlen, trainable=False))
model.add(Conv1D(16, 7, activation='relu'))
model.add(MaxPooling1D(pool_size=2))
model.add(LSTM(32, return_sequences=True))
model.add(LSTM(64))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_test, y_test))

y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print("Precision:", precision)
print("Recall:", recall)

test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/55852 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/13964 [00:00<?, ? examples/s]

Epoch 1/5
1397/1397 [==============================] - 108s 73ms/step - loss: 0.6873 - accuracy: 0.5476 - val_loss: 0.6838 - val_accuracy: 0.5486
Epoch 2/5
1397/1397 [==============================] - 87s 63ms/step - loss: 0.6837 - accuracy: 0.5517 - val_loss: 0.6839 - val_accuracy: 0.5520
Epoch 3/5
1397/1397 [==============================] - 87s 63ms/step - loss: 0.6826 - accuracy: 0.5566 - val_loss: 0.6837 - val_accuracy: 0.5501
Epoch 4/5
1397/1397 [==============================] - 91s 65ms/step - loss: 0.6822 - accuracy: 0.5570 - val_loss: 0.6825 - val_accuracy: 0.5521
Epoch 5/5
350/350 [==============================] - 6s 15ms/step
Confusion Matrix:
[[2527 3122]
 [1869 3653]]
Precision: 0.5391881918819188
Recall: 0.6615356754798986
350/350 [==============================] - 6s 18ms/step - loss: 0.6823 - accuracy: 0.5532
Test Loss: 0.6822782158851624
Test Accuracy: 0.5532181262969971


#Third dataset

In [7]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, LSTM, Dense
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from datasets import load_dataset
from sklearn.model_selection import train_test_split

dataset = load_dataset("saied/persian_news_dataset", split="train", streaming=True)

docs = []
document_count = 1000
counter = 0
skip_counter = 0

for doc in dataset:
    if len(doc['title']) == 0 or len(doc['category']) == 0:
        skip_counter += 1
        if skip_counter % 10000 == 0:
            print(f'{skip_counter} skipped')
        continue
    docs.append(doc)
    counter += 1
    if counter == document_count:
        break

texts = [doc['text'] for doc in docs]
categories = [doc['category'] for doc in docs]

max_words = 10000
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

input_length = 50
X = pad_sequences(sequences, maxlen=input_length)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(categories)
number_of_classes = len(label_encoder.classes_)
y = tf.keras.utils.to_categorical(y, num_classes=number_of_classes)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

glove_embeddings_index = {}
with open('/content/gdrive/MyDrive/glove.6B/glove.6B.100d.txt', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        glove_embeddings_index[word] = coefs

embedding_dim = 100
word_index = tokenizer.word_index
num_words = min(max_words, len(word_index) + 1)
embedding_matrix = np.zeros((num_words, embedding_dim))
for word, i in word_index.items():
    if i >= max_words:
        continue
    embedding_vector = glove_embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

model = Sequential()
model.add(Embedding(input_dim=num_words, output_dim=embedding_dim, input_length=input_length))
model.add(Conv1D(32, 7, activation='relu'))
model.add(LSTM(32, return_sequences=True))
model.add(LSTM(64))
model.add(Dense(number_of_classes, activation='softmax'))

model.layers[0].set_weights([embedding_matrix])
model.layers[0].trainable = False

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_val, y_val), verbose=2)

val_loss, val_accuracy = model.evaluate(X_val, y_val, verbose=0)
y_pred = np.argmax(model.predict(X_val), axis=1)
y_true = np.argmax(y_val, axis=1)

precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')
cm = confusion_matrix(y_true, y_pred)

print(f"Validation Loss: {val_loss}")
print(f"Validation Accuracy: {val_accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")
print("Confusion Matrix:")
print(cm)

/usr/local/lib/python3.10/dist-packages/datasets/load.py:1491: FutureWarning: The repository for saied/persian_news_dataset contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/saied/persian_news_dataset
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


10000 skipped
20000 skipped
30000 skipped
40000 skipped
50000 skipped
60000 skipped
70000 skipped
80000 skipped
90000 skipped
100000 skipped
110000 skipped
120000 skipped
130000 skipped
140000 skipped
150000 skipped
160000 skipped
170000 skipped
180000 skipped
190000 skipped
200000 skipped
210000 skipped
Epoch 1/5
25/25 - 6s - loss: 1.6742 - accuracy: 0.2887 - val_loss: 1.6017 - val_accuracy: 0.2950 - 6s/epoch - 238ms/step
Epoch 2/5
25/25 - 2s - loss: 1.4377 - accuracy: 0.4638 - val_loss: 1.3977 - val_accuracy: 0.4550 - 2s/epoch - 65ms/step
Epoch 3/5
25/25 - 2s - loss: 1.2952 - accuracy: 0.5125 - val_loss: 1.3128 - val_accuracy: 0.5000 - 2s/epoch - 65ms/step
Epoch 4/5
25/25 - 2s - loss: 1.1580 - accuracy: 0.5750 - val_loss: 1.2655 - val_accuracy: 0.5100 - 2s/epoch - 66ms/step
Epoch 5/5
25/25 - 1s - loss: 1.1173 - accuracy: 0.5863 - val_loss: 1.2305 - val_accuracy: 0.5350 - 1s/epoch - 41ms/step
7/7 [==============================] - 1s 14ms/step
Validation Loss: 1.2305233478546143
Valid

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
